## 🚨 URGENT 🚨 

## RQ6609 - 2026 Geolocation and FTE data for SRS Policy

Last updated by AF -> 20th October 2026 

### SDPT Allocation Details

| **Agreed Release Date** | 2nd November 2026 |
|---------|---------|
| **Received** | 22nd September 2026 |
| **Previous RQ** | RQ6204 |
| **Assigned to** | Matthew Gibbins |
| **Reviewer(s)** | Anthony Frost|
| **Completed by Date** | TBC |
| **Time to Complete** | TBC |
| **Review By Date** | TBC |
| **Save Folder** | "O:\Schools Data\6. Data\6.2 Data Requests\RQ6001-7000\RQ6601-6700\RQ6609 2026 Geolocation and FTE data for SRS Policy" |

### Requestor Details

| **Name** | Olivia Heys |
|---------|---------|
| **Email** | Olivia.Heys@education.gov.au |
| **Phone** | (02) 6218 4009 |
| **Job Title (Position)** | Policy Officer |
| **Team Name** | Schooling Resource Standard (SRS) Policy |
| **Organisation Name** | Department of Education |

### Requested Data Items

<br>

| **Filters & Aggregations** | | | | **Data Elements** | | |
|---------|---------|---|---|---------|---------|---|
| **Type** | **Values** |  | # | **Category** | **Name** | **Source** |
| Cohort | All |  | 1 | School | School AGEID | pr_dap_efs.business_sdpt_nongov_census.`schools_geography_2026` <br> [AGEID] |
| Sector | All → Gov & Non-gov |  | 2 | School Campus | Location AGEID | pr_dap_efs.business_sdpt_nongov_census.`schools_geography_2026` <br> [School_Location_AGEID] |
| Reporting Year | 2026 |  | 3 | School Campus | Main Location Flag | pr_dap_efs.business_sdpt_nongov_census.`schools_geography_2026` <br> [IsPrimaryLocation] |
| Student Data | FTE -> funded & FFPOS |  | 4 | Geographic | Latitude & Longitude | pr_dap_efs.business_sdpt_nongov_census.`schools_geography_2026` <br> [IGas.Latitude, IGas.Longitude] |
| School Data | Open Schools Only |  | 5 | Geographic | ABS SA1 Code | pr_dap_efs.business_sdpt_nongov_census.`schools_geography_2026` <br> [SA12021.SA1_CODE21] |
| Education Level | All |  | 6 | Student | Funded FTE | pr_dap_efs.efs_gold.`ngc_student_cohort_extract` <br> [fte] <br><br> pr_dap_efs.business_sdpt_gov_census.`2026_gsds_output_???_enrolment_prelim` <br> {where "???" = state abbreviation} <br> [FTE] |
| Aggregation Level | by School Campus |  | 7 | Student | FFPOS FTE | pr_dap_efs.efs_gold.`ngc_student_cohort_extract` <br> [overseas_student] <br><br> pr_dap_efs.business_sdpt_gov_census.`2026_gsds_output_???_enrolment_prelim` <br> {where "???" = state abbreviation} <br> [OS_S500(FFPOS)] |

### Output Dataset Structure

| # | **Name** | **id** | **Type** | **Description** |
|---|----------|----------|----------|----------|
| 1 | Year | year | int | Calendar year. |
| 2 | School AGEID | school_ageid | int | Australian Government Education Identifier (AGEID) of the school. |
| 3 | Location AGEID | location_ageid | int | Australian Government Education Identifier (AGEID) of the location (campus). |
| 4 | Main Location | main_location | str | Indicates whether the location is the main location of the school. <br> "Yes", "No" |
| 5 | Latitude | latitude | dbl | Geographic latitude of the location measured in decimal degrees north or south of the equator. |
| 6 | Longitude | longitude | dbl | Geographic longitude of the location measured in decimal degrees east or west of the Greenwich Meridian. |
| 7 | ABS SA1 | abs_sa1 | str | Statistical Area 1 code as defined by the ABS Australian Statistical Geography Standard 2021. |
| 8 | FTE  | funded_fte | dbl | The total full time equivalent (FTE) funded students at the school campus, including Indigenous students. Does not include Distance Education and Full Fee Paying Overseas Students. |
| 9 | FFPOS | ffpos_fte | dbl | Reported number of full fee paying overseas students (FTE) |

## Initial Notebook Setup  


In [0]:
# load the required library packages
from pyspark.sql import functions as F
from pyspark.sql.functions import monotonically_increasing_id

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

from pyspark.sql.functions import col, sum, when, count

from functools import reduce

In [0]:
%skip
# parameterise Catalog & Schema to dynamically update when required
dbutils.widgets.text("catalog", "xp_dap_efs")
dbutils.widgets.text("schema1", "business_sdpt_nongov_census")
dbutils.widgets.text("geopgraphy_table", "schools_geography_2026")

dbutils.widgets.text("schema2", "efs_gold")
dbutils.widgets.text("enrolment_table", "ngc_student_cohort")

# parameterise dataframe name to dynamically update when required
dbutils.widgets.text("final_dataset", "schools_df")

## Populate Initial Schools Data

In [0]:
# load the 2026 school data from new geography table filtering for any school which has a closed date
# this should mean we keep open schools only in the list.
schools_df = (
    spark.table("pr_dap_efs.business_sdpt_nongov_census.Schools_Geography_2026")
    .filter(
        F.col("School_location_ClosingDate").isNull()
        & F.col("school_location_EffectiveToDate").isNull()
        & (F.col("school_location_StatusID") == 1)
    )
    .select(
        F.lit(2026).alias("year"),
        F.col("AGEID").cast("int").alias("school_ageid"),
        F.col("School_location_AGEID").cast("int").alias("location_ageid"),
        F.col("IsPrimaryLocation").alias("main_location"),
        F.col("`IGas.Latitude`").alias("latitude"),
        F.col("`IGas.Longitude`").alias("longitude"),
        F.col("`SA12021.SA1_CODE21`").alias("abs_sa1")
    )
)

In [0]:
# update the true/false flag into human readable Yes/No
schools_df = schools_df.withColumn(
    "main_location",
    F.when(F.col("main_location").isNull(), None)
     .when(F.col("main_location"), "Yes")
     .otherwise("No")
)

In [0]:
# create a separate dataframe for duplicate location_ageid's 
duplicate_locations_df = (
    schools_df
    .groupBy("location_ageid")
    .count()
    .filter(F.col("count") > 1)
    .select("location_ageid")
)

schools_duplicates_df = (
    schools_df.alias("s")
    .join(
        duplicate_locations_df.alias("d"),
        on="location_ageid",
        how="inner"
    )
    .orderBy("location_ageid", "school_ageid")
)

# view duplicate dataframe
display(schools_duplicates_df)

location_ageid,year,school_ageid,main_location,latitude,longitude,abs_sa1


In [0]:
# compare all records that share the same location_ageid by grouping the rows for each 
# location and checking which columns contain more than one distinct value.
compare_cols = [
    c for c in schools_duplicates_df.columns
    if c != "location_ageid"
]

for col_name in compare_cols:
    diff_count = (
        schools_duplicates_df
        .groupBy("location_ageid")
        .agg(F.countDistinct(col_name).alias("cnt"))
        .filter(F.col("cnt") > 1)
        .count()
    )

    if diff_count > 0:
        print(f"{col_name}: {diff_count} location_ageid values have differences")
    else:
        schools_df = schools_df.dropDuplicates(["location_ageid"])

# clean up variables no longer required
del col_name, compare_cols, diff_count, duplicate_locations_df, schools_duplicates_df

In [0]:
# count the number of unique school locations (via location_ageid)
print(schools_df.count())

# view dataframe
display(schools_df)

10833


year,school_ageid,location_ageid,main_location,latitude,longitude,abs_sa1
2026,11122,5767,No,-35.3977255000000000,149.090799519999990,80107109011
2026,26766,6124,No,-9.3958932900000001,142.689956470000000,31501140210
2026,26766,6125,No,-9.4216998200000006,142.536169169999990,31501140207
2026,16073,6175,No,-37.6825542300000010,144.931329979999990,21005124224
2026,86076,6181,Yes,-37.5595172100000030,143.892109780000000,20101148112
2026,15519,6190,Yes,-37.7980876000000020,144.811082360000000,21301133826
2026,16370,6211,No,-37.8435754300000010,144.880428779999990,21302134404
2026,6230,6229,No,-37.9782495199999990,145.211275269999990,21204156303
2026,16383,6253,No,-37.7550898300000030,144.901721950000000,20603150114
2026,23412,6316,No,-37.7764668100000020,144.945591009999990,20604112401


## Populate Non-Governmnt FTE & FFPOS data

In [0]:
# pr_dap_efs.efs_gold.ngc_student_cohort_extract

student_cohort_df = (
    spark.table("pr_dap_efs.efs_gold.ngc_student_cohort_extract")
    .filter(F.col("CalendarYear") == 26)
    .groupBy("location_ageid")
    .agg(
        # total FTE
        F.sum(
            F.col("fte").cast("double") * F.col("headcount").cast("double")
        ).alias("total_fte"),

        # distance education FTE
        F.sum(
            F.when(
                F.col("education_type") == "Distance Education Student",
                F.col("fte").cast("double") * F.col("headcount").cast("double")
            )
        ).alias("de_fte"),

        # FFPOS FTE
        F.sum(
            F.when(
                F.col("overseas_student") == "Yes",
                F.col("fte").cast("double") * F.col("headcount").cast("double")
            )
        ).alias("ffpos_fte"),

        # students who are BOTH DE and FFPOS
        F.sum(
            F.when(
                (F.col("education_type") == "Distance Education Student")
                & (F.col("overseas_student") == "Yes"),
                F.col("fte").cast("double") * F.col("headcount").cast("double")
            )
        ).alias("de_ffpos_fte")
    )
    .fillna({
        "de_fte": 0,
        "ffpos_fte": 0,
        "de_ffpos_fte": 0
    })
    .withColumn(
        "funded_fte",
        F.col("total_fte")
        - F.col("de_fte")
        - F.col("ffpos_fte")
        + F.col("de_ffpos_fte")
    )
    .orderBy("location_ageid")
)

display(student_cohort_df)

location_ageid,total_fte,de_fte,ffpos_fte,de_ffpos_fte,funded_fte
40003,408.0,0.0,0.0,0.0,408.0
40004,416.0,0.0,0.0,0.0,416.0
40005,278.7,0.0,0.0,0.0,278.7
40007,380.8,0.0,0.0,0.0,380.8
40009,196.0,0.0,0.0,0.0,196.0
40010,97.0,0.0,0.0,0.0,97.0
40011,200.0,0.0,0.0,0.0,200.0
40012,522.0,0.0,14.0,0.0,508.0
40013,207.2,0.0,0.0,0.0,207.2
40018,399.0,0.0,0.0,0.0,399.0


In [0]:
# remove rows where these no data (all values are 0)
student_cohort_df = student_cohort_df.filter(
    ~(
        (F.col("total_fte") == 0) &
        (F.col("de_fte") == 0) &
        (F.col("ffpos_fte") == 0) &
        (F.col("de_ffpos_fte") == 0) &
        (F.col("funded_fte") == 0)
    )
)

student_cohort_df = (
    student_cohort_df
    .drop("de_fte", "de_ffpos_fte", "total_fte")
    .select(
        "location_ageid",
        "funded_fte",
        "ffpos_fte"
    )
)

display(student_cohort_df)

location_ageid,funded_fte,ffpos_fte
40003,408.0,0.0
40004,416.0,0.0
40005,278.7,0.0
40007,380.8,0.0
40009,196.0,0.0
40010,97.0,0.0
40011,200.0,0.0
40012,508.0,14.0
40013,207.2,0.0
40018,399.0,0.0


## Populate Governmnt FTE & FFPOS data 

In [0]:
# load the government enrolment data file
gov_enrolment_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(
             "file:/Workspace/Users/anthony.frost2@education.gov.au/Data Requests/RQ6619/2026 - All Schools/GSDS_enrolment_output_ALL_prelim.csv"
             
         )
         .select(
             F.col("Location_AGEID").alias("location_ageid"),
             F.col("FTE").alias("fte"),
             F.col("Headcount").alias("headcount"),
             F.col("DE").alias("education_type"),
             F.col("OS_S500(FFPOS)").alias("overseas_student"),
         )
)

display(gov_enrolment_df)

location_ageid,fte,headcount,education_type,overseas_student
44753,22.0,22,No,No
44753,1.0,1,No,No
44753,18.0,18,No,No
44753,23.0,23,No,No
44753,11.0,11,No,No
44753,1.0,1,No,No
44753,3.0,3,No,No
44753,16.0,16,No,No
44753,20.0,20,No,No
44753,1.0,1,No,No


In [0]:
gov_enrolment_df = (
    gov_enrolment_df
    .groupBy("location_ageid")
    .agg(
        # total FTE
        F.sum(
            F.col("fte").cast("double")
        ).alias("total_fte"),

        # distance education FTE
        F.sum(
            F.when(
                F.col("education_type") == "Yes",
                F.col("fte").cast("double")
            )
        ).alias("de_fte"),

        # FFPOS FTE
        F.sum(
            F.when(
                F.col("overseas_student") == "Yes",
                F.col("fte").cast("double")
            )
        ).alias("ffpos_fte"),

        # students who are BOTH DE and FFPOS
        F.sum(
            F.when(
                (F.col("education_type") == "Yes")
                & (F.col("overseas_student") == "Yes"),
                F.col("fte").cast("double")
            )
        ).alias("de_ffpos_fte")
    )
    .fillna({
        "de_fte": 0,
        "ffpos_fte": 0,
        "de_ffpos_fte": 0
    })
    .withColumn(
        "funded_fte",
        F.col("total_fte")
        - F.col("de_fte")
        - F.col("ffpos_fte")
        + F.col("de_ffpos_fte")
    )
    .filter(
        ~(
            (F.col("total_fte") == 0) &
            (F.col("de_fte") == 0) &
            (F.col("ffpos_fte") == 0) &
            (F.col("de_ffpos_fte") == 0) &
            (F.col("funded_fte") == 0)
        )
    )
    .drop("de_fte", "de_ffpos_fte", "total_fte")
    .select(
        "location_ageid",
        "funded_fte",
        "ffpos_fte"
    )
    .orderBy("location_ageid")
)

display(gov_enrolment_df)

location_ageid,funded_fte,ffpos_fte
5767,572.5,5.0
6124,56.0,0.0
6125,31.0,0.0
6175,358.0,1.0
6181,630.9,0.0
6190,767.9,24.0
6211,374.0,12.0
6253,346.1,46.0
6328,173.2,0.0
6354,594.0,0.0


In [0]:
# Append the two dataframes
combined_df = (
    student_cohort_df
    .unionByName(
        gov_enrolment_df,
        allowMissingColumns=True
    )
)

display(combined_df)

location_ageid,funded_fte,ffpos_fte
40003,408.0,0.0
40004,416.0,0.0
40005,278.7,0.0
40007,380.8,0.0
40009,196.0,0.0
40010,97.0,0.0
40011,200.0,0.0
40012,508.0,14.0
40013,207.2,0.0
40018,399.0,0.0


In [0]:
schools_df = (
    schools_df.alias("s")
    .join(
        combined_df.select(
            "location_ageid",
            "funded_fte",
            "ffpos_fte"
        ).alias("c"),
        on="location_ageid",
        how="full"
    )
    .orderBy("location_ageid")
)

# delete temporary variables no longer required
del student_cohort_df, gov_enrolment_df, combined_df

In [0]:
# change any enrolment of fte values of 0 to null for quick filtering of rows 
schools_df = schools_df.select(
    *[
        F.when(F.col(c) == 0, None).otherwise(F.col(c)).alias(c)
        if c in [
            "funded_fte",
            "ffpos_fte"
        ]
        else F.col(c)
        for c in schools_df.columns
    ]
)

display(schools_df)

# NOTE: based on a directive from Poonam (email on 6/11/2026), code to remove schools with no enrolment data was removed
#       the result means there should be an additional 38 locations listed

"""
# filter out all rows (schools) which have no enrolment data
schools_df = schools_df.dropna(
    how="all",
    subset=[
        "funded_fte",
        "ffpos_fte"
    ]
)
"""

location_ageid,year,school_ageid,main_location,latitude,longitude,abs_sa1,funded_fte,ffpos_fte
5767,2026,11122,No,-35.3977255000000000,149.090799519999990,80107109011,572.5,5.0
6124,2026,26766,No,-9.3958932900000001,142.689956470000000,31501140210,56.0,null
6125,2026,26766,No,-9.4216998200000006,142.536169169999990,31501140207,31.0,null
6175,2026,16073,No,-37.6825542300000010,144.931329979999990,21005124224,358.0,1.0
6181,2026,86076,Yes,-37.5595172100000030,143.892109780000000,20101148112,630.9,null
6190,2026,15519,Yes,-37.7980876000000020,144.811082360000000,21301133826,767.9,24.0
6211,2026,16370,No,-37.8435754300000010,144.880428779999990,21302134404,374.0,12.0
6229,2026,6230,No,-37.9782495199999990,145.211275269999990,21204156303,null,null
6253,2026,16383,No,-37.7550898300000030,144.901721950000000,20603150114,346.1,46.0
6316,2026,23412,No,-37.7764668100000020,144.945591009999990,20604112401,null,null


'\n# filter out all rows (schools) which have no enrolment data\nschools_df = schools_df.dropna(\n    how="all",\n    subset=[\n        "funded_fte",\n        "ffpos_fte"\n    ]\n)\n'

In [0]:
# place columns & data in order
schools_df = schools_df.select(
    "year",
    "school_ageid",
    "location_ageid",
    "main_location",
    "latitude",
    "longitude",
    "abs_sa1",
    "funded_fte",
    "ffpos_fte"
).orderBy("location_ageid", "school_ageid")

# inspect dataframe
display(schools_df)

year,school_ageid,location_ageid,main_location,latitude,longitude,abs_sa1,funded_fte,ffpos_fte
2026,11122,5767,No,-35.3977255000000000,149.090799519999990,80107109011,572.5,5.0
2026,26766,6124,No,-9.3958932900000001,142.689956470000000,31501140210,56.0,null
2026,26766,6125,No,-9.4216998200000006,142.536169169999990,31501140207,31.0,null
2026,16073,6175,No,-37.6825542300000010,144.931329979999990,21005124224,358.0,1.0
2026,86076,6181,Yes,-37.5595172100000030,143.892109780000000,20101148112,630.9,null
2026,15519,6190,Yes,-37.7980876000000020,144.811082360000000,21301133826,767.9,24.0
2026,16370,6211,No,-37.8435754300000010,144.880428779999990,21302134404,374.0,12.0
2026,6230,6229,No,-37.9782495199999990,145.211275269999990,21204156303,null,null
2026,16383,6253,No,-37.7550898300000030,144.901721950000000,20603150114,346.1,46.0
2026,23412,6316,No,-37.7764668100000020,144.945591009999990,20604112401,null,null


## Fix Data Issues Identified during Validation

missing school campus data manually taken from EFS Prgoam Admin
<br>
missing SA1 values taken from ABS maps


#### location_ageid "50079"

In [0]:
schools_df = (
    schools_df
    .withColumn(
        "year",
        F.when(F.col("location_ageid") == 50079, F.lit(2026))
         .otherwise(F.col("year"))
    )
    .withColumn(
        "school_ageid",
        F.when(F.col("location_ageid") == 50079, F.lit(10079))
         .otherwise(F.col("school_ageid"))
    )
    .withColumn(
        "main_location",
        F.when(F.col("location_ageid") == 50079, F.lit("Yes"))
         .otherwise(F.col("main_location"))
    )
    .withColumn(
        "latitude",
        F.when(F.col("location_ageid") == 50079, F.lit(-35.149079))
         .otherwise(F.col("latitude"))
    )
    .withColumn(
        "longitude",
        F.when(F.col("location_ageid") == 50079, F.lit(150.696121))
         .otherwise(F.col("longitude"))
    )
    .withColumn(
        "abs_sa1",
        F.when(F.col("location_ageid") == 50079, F.lit(90103100301))
         .otherwise(F.col("abs_sa1"))
    )
)

#### location_ageid "101583"

In [0]:
schools_df = (
    schools_df
    .withColumn(
        "year",
        F.when(F.col("location_ageid") == 101583, F.lit(2026))
         .otherwise(F.col("year"))
    )
    .withColumn(
        "school_ageid",
        F.when(F.col("location_ageid") == 101583, F.lit(17497))
         .otherwise(F.col("school_ageid"))
    )
    .withColumn(
        "main_location",
        F.when(F.col("location_ageid") == 101583, F.lit("Yes"))
         .otherwise(F.col("main_location"))
    )
    .withColumn(
        "latitude",
        F.when(F.col("location_ageid") == 101583, F.lit(-36.36748))
         .otherwise(F.col("latitude"))
    )
    .withColumn(
        "longitude",
        F.when(F.col("location_ageid") == 101583, F.lit(145.415551))
         .otherwise(F.col("longitude"))
    )
    .withColumn(
        "abs_sa1",
        F.when(F.col("location_ageid") == 101583, F.lit(21603141640))
         .otherwise(F.col("abs_sa1"))
    )
)

#### location+ageid "86780"

In [0]:
schools_df = (
    schools_df
    .withColumn(
        "latitude",
        F.when(F.col("location_ageid") == 86780, F.lit(-27.585908))
         .otherwise(F.col("latitude"))
    )
    .withColumn(
        "longitude",
        F.when(F.col("location_ageid") == 86780, F.lit(153.293470))
         .otherwise(F.col("longitude"))
    )
    .withColumn(
        "abs_sa1",
        F.when(F.col("location_ageid") == 86780, F.lit(30102101333))
         .otherwise(F.col("abs_sa1"))
    )
)

#### Null Handling

ICSEA values missing for 15 schools/school locations should remain as null (or na/not available)

Enrolments and FTE values should be displayed as 0 rather than null

In [0]:
# update any null values to 0 for enrolments and fte
schools_df = schools_df.fillna(
    {
        "funded_fte": 0,
        "ffpos_fte": 0,
    }
)

#### Review dataframe 

In [0]:
# place columns & data in order
schools_df = schools_df.select(
    "year",
    "school_ageid",
    "location_ageid",
    "main_location",
    "funded_fte",
    "ffpos_fte",
    "latitude",
    "longitude",
    "abs_sa1"
).orderBy("location_ageid", "school_ageid")

# inspect dataframe
display(schools_df)

year,school_ageid,location_ageid,main_location,funded_fte,ffpos_fte,latitude,longitude,abs_sa1
2026,11122,5767,No,572.5,5.0,-35.3977255,149.09079952,80107109011
2026,26766,6124,No,56.0,0.0,-9.39589329,142.68995647,31501140210
2026,26766,6125,No,31.0,0.0,-9.42169982,142.53616917,31501140207
2026,16073,6175,No,358.0,1.0,-37.68255423,144.93132998,21005124224
2026,86076,6181,Yes,630.9,0.0,-37.55951721,143.89210978,20101148112
2026,15519,6190,Yes,767.9,24.0,-37.7980876,144.81108236,21301133826
2026,16370,6211,No,374.0,12.0,-37.84357543,144.88042878,21302134404
2026,6230,6229,No,0.0,0.0,-37.97824952,145.21127527,21204156303
2026,16383,6253,No,346.1,46.0,-37.75508983,144.90172195,20603150114
2026,23412,6316,No,0.0,0.0,-37.77646681,144.94559101,20604112401


## Dataset Validation

In [0]:
# use this function to display all the distint values of a column, the row count and total for confirmation
def display_value_counts(column_name):
    df = globals()[dbutils.widgets.get("df_to_validate")]

    df_counts = (
        df.groupBy(column_name)
          .agg(F.count("*").alias("Row_Count"))
    )

    df_total = (
        df.agg(F.count("*").alias("row_count"))
          .withColumn(column_name, F.lit("TOTAL"))
          .select(column_name, "row_count")
    )

    display(
        df_counts
            .unionByName(df_total)
            .orderBy(
                F.when(F.col(column_name) == "TOTAL", 1).otherwise(0),
                F.col(column_name)
            )
    )

In [0]:
# check the total number of columns in the final table 
# based on the data request and final structure we are expecting 18

# ============================================
# validation PASS when # of columns == 21
# ============================================
print(f"Number of columns: {len(globals()[dbutils.widgets.get("df_to_validate")].columns)} \n")

Number of columns: 9 



In [0]:
# desired column headings, directly copied from the sample dataset supplied
desired_columns = [
    "year",
    "school_ageid",
    "location_ageid",
    "main_location",
    "latitude",
    "longitude",
    "abs_sa1",
    "funded_fte",
    "ffpos_fte"
]

# columns from the generated dataset
actual_columns = globals()[dbutils.widgets.get("df_to_validate")].columns

# display side-by-side comparison
max_len = max(len(desired_columns), len(actual_columns))

comparison_data = [
    (
        desired_columns[i] if i < len(desired_columns) else None,
        actual_columns[i] if i < len(actual_columns) else None
    )
    for i in range(max_len)
]

comparison_df = spark.createDataFrame(
    comparison_data,
    ["Desired Column Headings", f"{dbutils.widgets.get('df_to_validate')} Column Headings"]
)

# ========================================================
# validation PASS when visual check confirms columns match
# ========================================================

# view the dataframe results
display(comparison_df)

# variables no longer required
del actual_columns, max_len, comparison_data, comparison_df, desired_columns

Desired Column Headings,schools_df Column Headings
year,year
school_ageid,school_ageid
location_ageid,location_ageid
main_location,main_location
latitude,funded_fte
longitude,ffpos_fte
abs_sa1,latitude
funded_fte,longitude
ffpos_fte,abs_sa1


In [0]:
# count nulls in each column
null_counts = globals()[dbutils.widgets.get("df_to_validate")].select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in globals()[dbutils.widgets.get("df_to_validate")].columns
])

# transpose null count for easier viewing
null_counts_long = null_counts.selectExpr(
    f"""
    stack(
        {len(globals()[dbutils.widgets.get("df_to_validate")].columns)},
        {', '.join([f"'{c}', `{c}`" for c in globals()[dbutils.widgets.get("df_to_validate")].columns])}
    ) as (ColumnName, NullCount)
    """
)

# view the dataframe results
display(null_counts_long)

# variables no longer required
del null_counts, null_counts_long

# =============================================
# validation PASS when all NullCount values = 0
# =============================================

ColumnName,NullCount
year,0
school_ageid,0
location_ageid,0
main_location,0
funded_fte,0
ffpos_fte,0
latitude,0
longitude,0
abs_sa1,0


In [0]:
# view all uniqe values of "year" and confirm they are accurate
# already confirmed there's no null values
display(
    globals()[dbutils.widgets.get("df_to_validate")]
        .select("year")
        .distinct()
        .orderBy("year")
)

# ================================================
# validation PASS when only returned value is 2026
# ================================================

year
2026


In [0]:
# The following columns are confirmed as having no null values:
#
# school_ageid

In [0]:
# check for any null location_ageid values
null_rows = globals()[dbutils.widgets.get("df_to_validate")].filter(col("location_ageid").isNull())
null_count = null_rows.count()

# check for duplicate location_ageid values
duplicate_ids = (
    globals()[dbutils.widgets.get("df_to_validate")]
    .groupBy("location_ageid")
    .count()
    .filter(col("count") > 1)
)

duplicate_count = duplicate_ids.count()

print(f"Null location_ageid count: {null_count}")
print(f"Duplicate location_ageid count: {duplicate_count}")

# display offending rows
if null_count > 0:
    print("Rows with NULL location_ageid:")
    display(null_rows)

if duplicate_count > 0:
    print("Rows with duplicate location_ageid values:")
    
    duplicate_rows = (
        globals()[dbutils.widgets.get("df_to_validate")].alias("a")
        .join(
            duplicate_ids.select("location_ageid").alias("b"),
            on="location_ageid",
            how="inner"
        )
        .orderBy("location_ageid")
    )
    
    # view the dataframe results
    display(duplicate_rows)

# overall validation result
if null_count == 0 and duplicate_count == 0:
    print("PASS: location_ageid contains no nulls and all values are unique.")
else:
    print("FAIL: Validation checks failed.")

# variables no longer required
del null_rows, null_count, duplicate_ids, duplicate_count

# ===============================================
# validation PASS when no nulls and no duplicates
# ===============================================

Null location_ageid count: 0
Duplicate location_ageid count: 0
PASS: location_ageid contains no nulls and all values are unique.


In [0]:
# view all unique values of "school_sector" and confirm they are accurate
# already confirmed there's no null values
display_value_counts("main_location")

# ============================================================
# validation PASS when only returned values are "Yes" or "No" 
# ============================================================

main_location,Row_Count
No,1019
Yes,9816
TOTAL,10835


In [0]:
# Approximate Australian geographic bounds
# Latitude:  -44 to -10
# Longitude: 112 to 154

display(
    schools_df.filter(
        (F.col("latitude").isNull()) |
        (F.col("longitude").isNull()) |
        (F.col("latitude") < -44) |
        (F.col("latitude") > -10) |
        (F.col("longitude") < 112) |
        (F.col("longitude") > 154)
    )
)

# ============================================================
# validation PASS when no rows are returned for viewing 
# ============================================================


# Display duplicate latitude/longitude pairs and how many records share them
duplicate_coords = (
    schools_df
        .groupBy("latitude", "longitude")
        .count()
        .filter(F.col("count") > 1)
        .select("latitude", "longitude")
)

display(
    schools_df
        .join(
            duplicate_coords,
            on=["latitude", "longitude"],
            how="inner"
        )
        .orderBy("latitude", "longitude", "location_ageid")
)

# delete temporary variables no longer required
del duplicate_coords

# ============================================================
# validation PASS after review of any duplicate pairs
# ============================================================

# NOTE: 13 school campuses fall outside the set threshold for lat/long values
#       a manual check confirms these are valid schools on islands outside mainland Australia
#       such as Norfolk Island, Coco (Keeling) Islands & Far North Queensland Islands

# NOTE: many schools share the exact same lat/long values -> 3,222
#       a random check of a handful of schools found that the values are "reasonable"
#       for example school_ageid's: 18162, 1255, 12024, 12029, 16353 share identical values
#       all these schools are in "Sale, NSW" so the values are close, but not precise
#       EFS appears to be more accurate

year,school_ageid,location_ageid,main_location,funded_fte,ffpos_fte,latitude,longitude,abs_sa1
2026,26766,6124,No,56.0,0.0,-9.39589329,142.68995647,31501140210
2026,26766,6125,No,31.0,0.0,-9.42169982,142.53616917,31501140207
2026,26766,13156,No,37.0,0.0,-9.58688072,143.77034566,31501140204
2026,26766,13158,No,53.0,0.0,-9.95488669,142.18156809,31501140212
2026,26766,13160,No,8.0,0.0,-9.50747921,143.54535039,31501140209
2026,26766,13164,No,38.0,0.0,-9.90101058,142.7749961,31501140201
2026,26766,13165,No,36.5,0.0,-9.7523019,143.40786623,31501140216
2026,26766,15073,No,36.5,0.0,-9.26483575,142.2152163,31501140206
2026,26766,18274,No,46.0,0.0,-9.91663593,144.05137067,31501140205
2026,11116,51116,Yes,36.0,0.0,-31.55332745,159.07684025,10803116101


latitude,longitude,year,school_ageid,location_ageid,main_location,funded_fte,ffpos_fte,abs_sa1
-43.02374,147.04692504,2026,6610,46610,Yes,267.91100000000006,0.0,60302107218
-43.02374,147.04692504,2026,8189,48189,Yes,247.0,0.0,60302107218
-42.94293485,147.354933,2026,6794,46794,Yes,1098.721,15.0,60104102608
-42.94293485,147.354933,2026,8067,48067,Yes,243.4,0.0,60104102608
-42.84946697,147.28125309,2026,55,40055,Yes,268.0,0.0,60103102105
-42.84946697,147.28125309,2026,30329,88127,Yes,139.4,0.0,60103102102
-42.83222058,147.29249744,2026,8073,48073,Yes,79.6,1.0,60103101608
-42.83222058,147.29249744,2026,29034,69034,Yes,0.0,0.0,60103101608
-42.77468018,147.25295755,2026,15939,55939,Yes,596.0,0.0,60103101305
-42.77468018,147.25295755,2026,90041,90042,Yes,5.0,0.0,60103101305


In [0]:
# list of government census data tables by state
tables = [
    "pr_dap_efs.business_sdpt_gov_census.2026_gsds_output_act_enrolment_prelim",
    "pr_dap_efs.business_sdpt_gov_census.2026_gsds_output_nsw_enrolment_prelim",
    "pr_dap_efs.business_sdpt_gov_census.2026_gsds_output_nt_enrolment_prelim",
    "pr_dap_efs.business_sdpt_gov_census.2026_gsds_output_qld_enrolment_prelim",
    "pr_dap_efs.business_sdpt_gov_census.2026_gsds_output_sa_enrolment_prelim",
    "pr_dap_efs.business_sdpt_gov_census.2026_gsds_output_tas_enrolment_prelim",
    "pr_dap_efs.business_sdpt_gov_census.2026_gsds_output_vic_enrolment_prelim",
    "pr_dap_efs.business_sdpt_gov_census.2026_gsds_output_wa_enrolment_prelim"
]

# calculate gov funded_fte
gov_funded_fte = 0.0

for table_name in tables:
    funded_fte = (
        spark.table(table_name)
        .agg(
            (
                F.sum("fte")
                - F.sum(F.when(F.col("DE") == "Yes", F.col("fte")).otherwise(0))
                - F.sum(F.when(F.col("OS_S500(FFPOS)") == "Yes", F.col("fte")).otherwise(0))
                + F.sum(
                    F.when(
                        (F.col("DE") == "Yes") &
                        (F.col("OS_S500(FFPOS)") == "Yes"),
                        F.col("fte")
                    ).otherwise(0)
                )
            ).alias("funded_fte")
        )
        .first()["funded_fte"]
    )

    gov_funded_fte += float(funded_fte or 0)

print(f"total_gov_funded_fte = {gov_funded_fte:,.2f}")



# calculate non-gov funded_fte
non_gov_funded_fte = (
    spark.table("pr_dap_efs.efs_gold.ngc_student_cohort_extract")
    .filter(F.col("CalendarYear") == 26)
    .agg(
        (
            F.sum(F.col("fte") * F.col("headcount"))
            - F.sum(
                F.when(
                    F.col("education_type") == "Distance Education Student",
                    F.col("fte") * F.col("headcount")
                ).otherwise(0)
            )
            - F.sum(
                F.when(
                    F.col("overseas_student") == "Yes",
                    F.col("fte") * F.col("headcount")
                ).otherwise(0)
            )
            + F.sum(
                F.when(
                    (F.col("education_type") == "Distance Education Student")
                    & (F.col("overseas_student") == "Yes"),
                    F.col("fte") * F.col("headcount")
                ).otherwise(0)
            )
        ).alias("total_non_gov_funded_fte")
    )
    .first()["total_non_gov_funded_fte"]
)

print(f"total_non_gov_funded_fte = {float(non_gov_funded_fte):,.2f}")

calc_total_funded_fte = float(gov_funded_fte) + float(non_gov_funded_fte)

print(f"\ncalc_total_funded_fte = {calc_total_funded_fte:,.2f}")


total_funded_fte = (
    schools_df
    .agg(F.sum("funded_fte").alias("total_funded_fte"))
    .first()["total_funded_fte"]
)

print(f"sum_total_funded_fte = {float(total_funded_fte):,.2f}")

# delete temporary variables no longer required
del non_gov_funded_fte, gov_funded_fte, calc_total_funded_fte, total_funded_fte, tables, table_name, funded_fte

# ============================================================
# validation PASS after when calulated fte == sum of fte
# ============================================================

total_gov_funded_fte = 2,555,565.81
total_non_gov_funded_fte = 1,555,813.90

calc_total_funded_fte = 4,111,379.71
sum_total_funded_fte = 4,111,379.71


In [0]:
# these are the columns to summarise
metrics = [
    "funded_fte",
    "ffpos_fte"
]

summary_rows = []

for metric in metrics:
    stats = (
        schools_df.select(
            F.min(metric).alias("Minimum"),
            F.max(metric).alias("Maximum"),
            F.expr(f"percentile_approx({metric}, 0.5)").alias("Median"),
            F.avg(metric).alias("Average"),
            F.sum(metric).alias("Total")
        )
        .collect()[0]
    )

    summary_rows.append(
        (
            metric,
            round(float(stats["Minimum"]), 4) if stats["Minimum"] is not None else None,
            round(float(stats["Maximum"]), 4) if stats["Maximum"] is not None else None,
            round(float(stats["Median"]), 4) if stats["Median"] is not None else None,
            round(float(stats["Average"]), 4) if stats["Average"] is not None else None,
            round(float(stats["Total"]), 4) if stats["Total"] is not None else None,
        )
    )

summary_df = spark.createDataFrame(
    summary_rows,
    [
        "Metric",
        "Minimum",
        "Maximum",
        "Median",
        "Average",
        "Total"
    ]
)

display(summary_df)

# delete temporary variables no longer required
del metric, metrics, summary_rows, summary_df, stats 

# ===============================================================
# validation PASS after a manual sanity check of the below values 
# ===============================================================

Metric,Minimum,Maximum,Median,Average,Total
funded_fte,0.0,3998.0,263.0,379.4536,4111379.71
ffpos_fte,0.0,221.0,0.0,2.2642,24532.3


## Complete metadata and finalise

In [0]:
print(f"Number of rows: {schools_df.count():,}")
schools_df.select(F.countDistinct("location_ageid").alias("distinct_location_ageid")).show()
display(schools_df)

Number of rows: 10,835
+-----------------------+
|distinct_location_ageid|
+-----------------------+
|                  10835|
+-----------------------+



year,school_ageid,location_ageid,main_location,funded_fte,ffpos_fte,latitude,longitude,abs_sa1
2026,11122,5767,No,572.5,5.0,-35.3977255,149.09079952,80107109011
2026,26766,6124,No,56.0,0.0,-9.39589329,142.68995647,31501140210
2026,26766,6125,No,31.0,0.0,-9.42169982,142.53616917,31501140207
2026,16073,6175,No,358.0,1.0,-37.68255423,144.93132998,21005124224
2026,86076,6181,Yes,630.9,0.0,-37.55951721,143.89210978,20101148112
2026,15519,6190,Yes,767.9,24.0,-37.7980876,144.81108236,21301133826
2026,16370,6211,No,374.0,12.0,-37.84357543,144.88042878,21302134404
2026,6230,6229,No,0.0,0.0,-37.97824952,145.21127527,21204156303
2026,16383,6253,No,346.1,46.0,-37.75508983,144.90172195,20603150114
2026,23412,6316,No,0.0,0.0,-37.77646681,144.94559101,20604112401


In [0]:
%sql
-- Independent source recomputation; no notebook dataframes or views required.
-- The SQL compute must be able to read the government CSV at the path below.
-- Expected for the reviewed snapshot: 38 locations (8 non-government, 30 government).
WITH source_rows AS (
    SELECT
        'Non-government' AS sector,
        CAST(location_ageid AS BIGINT) AS location_ageid,
        CAST(fte AS DOUBLE) * CAST(headcount AS DOUBLE) AS source_fte,
        education_type = 'Distance Education Student' AS is_de,
        overseas_student = 'Yes' AS is_ffpos
    FROM pr_dap_efs.efs_gold.ngc_student_cohort_extract
    WHERE CalendarYear = 26

    UNION ALL

    SELECT
        'Government' AS sector,
        CAST(Location_AGEID AS BIGINT) AS location_ageid,
        CAST(FTE AS DOUBLE) AS source_fte,
        DE = 'Yes' AS is_de,
        `OS_S500(FFPOS)` = 'Yes' AS is_ffpos
    FROM read_files(
        'file:/Workspace/Users/dmytro.yalovoy@education.gov.au/Data Requests/RQ6619/2026 - All Schools/GSDS_enrolment_output_ALL_prelim.csv',
        format => 'csv',
        header => true
    )
), fte_by_location AS (
    SELECT
        sector,
        location_ageid,
        SUM(source_fte) AS total_fte,
        COALESCE(SUM(CASE WHEN is_de THEN source_fte END), 0) AS de_fte,
        COALESCE(SUM(CASE WHEN is_ffpos THEN source_fte END), 0) AS ffpos_fte,
        COALESCE(SUM(CASE WHEN is_de AND is_ffpos THEN source_fte END), 0) AS de_ffpos_fte
    FROM source_rows
    GROUP BY sector, location_ageid
), combined_df AS (
    SELECT
        sector,
        location_ageid,
        total_fte - de_fte - ffpos_fte + de_ffpos_fte AS funded_fte,
        ffpos_fte
    FROM fte_by_location
    WHERE NOT (
        total_fte = 0
        AND de_fte = 0
        AND ffpos_fte = 0
        AND de_ffpos_fte = 0
        AND total_fte - de_fte - ffpos_fte + de_ffpos_fte = 0
    )
), zero_fte AS (
    SELECT sector, location_ageid, funded_fte, ffpos_fte
    FROM combined_df
    WHERE funded_fte = 0 AND ffpos_fte = 0
)
SELECT
    f.sector,
    f.location_ageid,
    f.funded_fte,
    f.ffpos_fte,
    (
        SELECT SORT_ARRAY(COLLECT_SET(CAST(g.AGEID AS BIGINT)))
        FROM pr_dap_efs.business_sdpt_nongov_census.schools_geography_2026 AS g
        WHERE CAST(g.School_location_AGEID AS BIGINT) =
              CAST(f.location_ageid AS BIGINT)
          AND g.School_location_ClosingDate IS NULL
          AND g.School_location_EffectiveToDate IS NULL
          AND g.School_location_StatusID = 1
    ) AS school_ageids
FROM zero_fte AS f
ORDER BY f.location_ageid;

sector,location_ageid,funded_fte,ffpos_fte,school_ageids
Government,6124,56.0,0.0,List(26766)
Government,6125,31.0,0.0,List(26766)
Government,6181,630.9,0.0,List(86076)
Government,6328,173.2,0.0,List(30197)
Government,6354,594.0,0.0,List(6360)
Government,6432,493.0,0.0,List(16647)
Government,6451,491.0,0.0,List(17696)
Government,6497,781.0,0.0,List(6493)
Government,6567,361.0,0.0,List(26766)
Government,7704,766.4000000000001,0.0,List(7710)



#### RQ6609 Metadata Table

| Data Item Name | Data Item Type | Data Item Length | Data Item Description / Meaning / Comments / Business Rules | Data Values |
|---|---|---|---|---|
| **School AGEID** | Integer | 5 | Australian Government Education ID (AGEID) of school. | Discrete |
| **Location AGEID** | Integer | 5 | Australian Government Education ID (AGEID) of school location. | Unique, Discrete |
| **Main Location** | Character | 3 | Indicates the main location of the school. | Yes, No |
| **Funded Student FTE** | Decimal | (5,1) | Full-Time Equivalent (FTE) of funded student enrolments. | Continuous |
| **FFPOS FTE** | Decimal | (5,1) | Full-Time Equivalent (FTE) of Full Fee Paying Overseas Students (FFPOS). | Continuous |
| **Latitude** | Decimal | (9,6) | Latitude of the school location as reported on EFS. | Continuous |
| **Longitude** | Decimal | (9,6) | Longitude of the school location as reported on EFS. | Continuous |
| **Statistical Area 2021** | Integer | 11 | Australian Bureau of Statistics (ABS) Australian Statistical Geography Standard (ASGS) Statistical Area Level 1 (SA1) 2021 code. An 11-digit identifier representing the statistical area in which the school is located. | Discrete |

### RQ6609 Notes

1. Latitude and Longitude of the school location are as reported in EFS.
2. ABS Statistical Area 2021 is obtained from the Boundary Selection in the Spatial Enablement Add-in using the latitude/longitude as reported in EFS.
3. Where the latitude and longitude is missing the SA1 2021 is obtained from the location address passed through the Spatial Add-in.
4. Funded Student FTE is the total student FTE excluding FFPOS FTE and Distance Education students
5. FFPOS FTE is the FTE of Full Fee Paying Oversas Students identified where the census response for overseas_student is 'Yes'. These are excluded from the student FTE in the Funded FTE total.
